# Pandas AI Functions Starter Notebook

Use pandas AI Functions on customer-review dataset to build a transformation workflow.

## What You'll Do
1. Set shared defaults with `aifunc.default_conf`, including `api_type="chat_completions"`.
2. Load a public review sample and inspect rating and review-length patterns before enrichment.
3. Walk through all nine Fabric AI Functions in one customer-feedback workflow.
4. Build a compact dashboard and review common pandas configuration options.

## Before You Start
- **Runtime**: Fabric Runtime 1.3 or later.
- **Temporary dependency**: Pandas AI Functions currently require the `openai` Python package in the notebook session.
- **Cost control**: Start with `SAMPLE_ROWS`, then scale up after you are happy with output quality.

| Resource | Link |
|---|---|
| AI Functions overview | [aka.ms/ai-functions](https://aka.ms/ai-functions) |
| Pandas configuration docs | [Pandas AI Function Config Doc](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/configuration) |
| Model catalog | [aka.ms/fabric-ai-models](https://aka.ms/fabric-ai-models) |
| More starter notebook examples | [aka.ms/fabric-aifunctions-starter-notebooks](https://aka.ms/fabric-aifunctions-starter-notebooks) |
| More eval notebook examples | [aka.ms/fabric-aifunctions-eval-notebooks](https://aka.ms/fabric-aifunctions-eval-notebooks) |

## 1. Imports and Installs
pandas AI Functions currently requires the `openai` python package to be installed in the notebook session

In [ ]:
%pip install -q openai 2>/dev/null

In [ ]:
import json
from textwrap import dedent
from typing import Literal

import matplotlib.pyplot as plt
import pandas as pd
import synapse.ml.aifunc as aifunc
from datasets import load_dataset
from pydantic import BaseModel, Field

SAMPLE_ROWS = 20

In [ ]:
# Shared chart styling and color palette
plt.style.use("seaborn-v0_8-whitegrid")

FABRIC_BLUE = "#4C78A8"
FABRIC_TEAL = "#72B7B2"
FABRIC_ORANGE = "#F58518"
FABRIC_RED = "#E45756"
FABRIC_GRAY = "#9D9DA1"
SENTIMENT_COLORS = {
    "positive": "#54A24B",
    "negative": FABRIC_RED,
    "neutral": FABRIC_GRAY,
    "mixed": FABRIC_ORANGE,
}
PRIORITY_COLORS = {
    "high": FABRIC_RED,
    "medium": FABRIC_ORANGE,
    "low": FABRIC_TEAL,
}

### 1.1 Shared defaults and targeted overrides

Pandas AI Functions support two configuration layers:
- `aifunc.default_conf` for shared session defaults
- `aifunc.Conf(...)` for targeted per-call overrides

Use `aifunc.default_conf` for workflow-wide settings. This walkthrough keeps `api_type`, `model_deployment_name`, and `temperature` fixed across the main steps, and uses `progress_bar_mode="stats"` to show token counts and CU estimates. Use `ai.<ai_function_name>(..., conf=aifunc.Conf(...))` only when a specific step needs a focused override, such as the evaluation pass later in the notebook.

In [ ]:
# Global defaults for the walkthrough.
aifunc.default_conf.api_type = "chat_completions"  # Use the chat completions API type
aifunc.default_conf.model_deployment_name = "gpt-4.1-mini"  # LLM powering the AI functions
aifunc.default_conf.embedding_deployment_name = "text-embedding-ada-002"  # Embedding model powering ai.embed and ai.similarity
aifunc.default_conf.temperature = 0.0  # Deterministic outputs for repeatable results.
aifunc.default_conf.progress_bar_mode = "stats"  # Show token counts and CU estimates in the progress bar.

display(
    pd.DataFrame(
        {
            "setting": [
                "api_type",
                "model_deployment_name",
                "embedding_deployment_name",
                "temperature",
                "concurrency",
                "progress_bar_mode",
            ],
            "value": [
                aifunc.default_conf.api_type,
                aifunc.default_conf.model_deployment_name,
                aifunc.default_conf.embedding_deployment_name,
                aifunc.default_conf.temperature,
                aifunc.default_conf.concurrency,
                aifunc.default_conf.progress_bar_mode,
            ],
            "why_it_matters": [
                "Chooses the model API surface for all notebook calls",
                "Controls which text model runs the AI functions",
                "Controls which embedding model powers ai.embed and ai.similarity",
                "Keeps outputs deterministic for a starter workflow",
                "Sets the maximum number of rows processed asynchronously",
                "`stats` shows token counts and CU estimates during execution",
            ],
        }
    )
)

## 2. Load data and profile the sample

We use a small public sample from Amazon Reviews 2023 (Sports and Outdoors) to keep iteration fast. The quick profile below helps you understand the rating mix and review length before AI enrichment.

In [ ]:
reviews = load_dataset(
    "McAuley-Lab/Amazon-Reviews-2023",
    "raw_review_Sports_and_Outdoors",
    split="full",
    streaming=True,
    trust_remote_code=True,
)

raw_df = pd.DataFrame(list(reviews.take(1000)))
df = raw_df.sample(n=min(SAMPLE_ROWS, len(raw_df)), random_state=2026).reset_index(drop=True)
df = df.rename(columns={"text": "review_text", "title": "review_title"})
df = df[["review_title", "review_text", "rating"]].copy()
df["review_title"] = df["review_title"].fillna("").astype(str)
df["review_text"] = df["review_text"].fillna("").astype(str)

load_info = pd.DataFrame(
    {
        "metric": ["rows_loaded", "columns", "dataset_config"],
        "value": [len(df), ", ".join(df.columns), "raw_review_Sports_and_Outdoors"],
    }
)
display(load_info)
display(df.head(10))

In [ ]:
df["review_char_count"] = df["review_text"].astype(str).str.len()

rating_counts = (
    df["rating"]
    .fillna("missing")
    .astype(str)
    .value_counts()
    .rename_axis("rating")
    .reset_index(name="count")
    .sort_values("rating")
)

eda_summary = pd.DataFrame(
    {
        "metric": [
            "row_count",
            "avg_review_char_count",
            "median_review_char_count",
            "missing_ratings",
        ],
        "value": [
            len(df),
            round(df["review_char_count"].mean(), 1),
            round(df["review_char_count"].median(), 1),
            int(df["rating"].isna().sum()),
        ],
    }
)

display(eda_summary)
display(rating_counts)
display(
    df[["rating", "review_char_count"]]
    .describe(include="all")
    .transpose()
    .reset_index()
    .rename(columns={"index": "column"})
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].bar(rating_counts["rating"], rating_counts["count"], color=FABRIC_BLUE, edgecolor="white")
axes[0].set_title("Rating distribution")
axes[0].set_xlabel("rating")
axes[0].set_ylabel("count")

df["review_char_count"].plot.hist(
    ax=axes[1],
    bins=min(12, max(5, len(df) // 2)),
    color=FABRIC_TEAL,
    edgecolor="white",
)
axes[1].axvline(
    df["review_char_count"].median(),
    color=FABRIC_ORANGE,
    linestyle="--",
    linewidth=2,
    label="median",
)
axes[1].set_title("Review length distribution")
axes[1].set_xlabel("characters")
axes[1].set_ylabel("count")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Walk through the AI Functions

This example workflow moves from signal detection -> enrichment -> action: start with sentiment and topic labels, extract structured facts, condense and clean the text, add semantic signals, and finish with support-ready outputs.

### 3.1 `ai.analyze_sentiment`

Start with sentiment so every downstream step has a first-pass triage signal.

In [ ]:
df["sentiment"] = df["review_text"].ai.analyze_sentiment()

display(df[["review_text", "rating", "sentiment"]].head(10))

In [ ]:
sentiment_order = ["positive", "neutral", "mixed", "negative"]
sentiment_counts = (
    df["sentiment"].astype(str).value_counts().rename_axis("sentiment").reset_index(name="count")
)
sentiment_counts["sentiment"] = pd.Categorical(
    sentiment_counts["sentiment"],
    categories=sentiment_order,
    ordered=True,
)
sentiment_counts = sentiment_counts.sort_values("sentiment").reset_index(drop=True)
display(sentiment_counts)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(
    sentiment_counts["sentiment"].astype(str),
    sentiment_counts["count"],
    color=[
        SENTIMENT_COLORS.get(value, FABRIC_BLUE)
        for value in sentiment_counts["sentiment"].astype(str)
    ],
    edgecolor="white",
)
ax.set_title("Sentiment distribution")
ax.set_xlabel("sentiment")
ax.set_ylabel("count")
plt.tight_layout()
plt.show()

### 3.2 `ai.classify`

Add a business topic so each review is easier to route to the right owner.

In [ ]:
TOPIC_LABELS = [
    "product_quality",
    "shipping_delivery",
    "customer_service",
    "price_value",
    "product_description",
]

df["topic"] = df["review_text"].ai.classify(*TOPIC_LABELS)

display(df[["review_text", "sentiment", "topic"]].head(10))

In [ ]:
topic_counts = df["topic"].astype(str).value_counts().rename_axis("topic").reset_index(name="count")
topic_counts = topic_counts.sort_values("count", ascending=True).reset_index(drop=True)

topic_sentiment = pd.crosstab(df["topic"].astype(str), df["sentiment"].astype(str)).reset_index()
display(topic_counts.sort_values("count", ascending=False).reset_index(drop=True))
display(topic_sentiment)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(topic_counts["topic"], topic_counts["count"], color=FABRIC_BLUE, edgecolor="white")
ax.set_title("Top classified topics")
ax.set_xlabel("count")
ax.set_ylabel("topic")
plt.tight_layout()
plt.show()

### 3.3 `ai.extract`

Extract product, brand, and issue details so the triage view contains concrete facts, not just labels.

In [ ]:
# Basic extraction with string labels
extracted = df["review_text"].ai.extract("product", "brand", "issue")
df = pd.concat([df, extracted], axis=1)

display(df[["review_text", "product", "brand", "issue"]].head(10))

#### 3.3.1 Typed extraction with `aifunc.ExtractLabel`

Use `aifunc.ExtractLabel` when you want field descriptions and expected types in the extraction schema.

In [ ]:
# Define typed extraction labels with descriptions
typed_labels = [
    aifunc.ExtractLabel(
        label="price_mentioned",
        description="Any price or cost mentioned in the review",
        type="string",
    ),
    aifunc.ExtractLabel(
        label="is_repeat_customer",
        description="Whether the reviewer indicates they have purchased before",
        type="boolean",
    ),
]

typed_extracted = df["review_text"].ai.extract(*typed_labels)
df = pd.concat([df, typed_extracted], axis=1)

display(df[["review_text", "price_mentioned", "is_repeat_customer"]].head(10))

In [ ]:
def has_entity(value):
    if value is None:
        return False
    if isinstance(value, list):
        return len(value) > 0
    if isinstance(value, float) and pd.isna(value):
        return False
    return str(value).strip() != ""

entity_summary = pd.DataFrame(
    {
        "entity": ["product", "brand", "issue", "price_mentioned"],
        "rows_with_values": [
            int(df["product"].apply(has_entity).sum()),
            int(df["brand"].apply(has_entity).sum()),
            int(df["issue"].apply(has_entity).sum()),
            int(df["price_mentioned"].apply(has_entity).sum()),
        ],
    }
).sort_values("rows_with_values", ascending=True)
display(entity_summary)

fig, ax = plt.subplots(figsize=(7, 4))
ax.barh(entity_summary["entity"], entity_summary["rows_with_values"], color=FABRIC_TEAL, edgecolor="white")
ax.set_title("Entity coverage across the sample")
ax.set_xlabel("rows with at least one extracted value")
ax.set_ylabel("entity")
plt.tight_layout()
plt.show()

#### 3.3.2 Evaluate and refine extracted issues with `ai.generate_response`

After extraction, add an evaluation column that reviews whether the extracted issue is ready for downstream triage and, when needed, produces a cleaner issue phrase. For richer judge-model patterns, multiple metrics, and side-by-side comparisons, see [aka.ms/fabric-aifunctions-eval-notebooks](https://aka.ms/fabric-aifunctions-eval-notebooks).

In [ ]:
EXTRACT_FIELDS = ["product", "brand", "issue"]

def stringify_entity(value):
    if not has_entity(value):
        return ""
    if isinstance(value, list):
        return ", ".join(str(item).strip() for item in value if str(item).strip())
    return str(value).strip()

def build_extraction_summary(row, fields):
    parts = []
    for field in fields:
        value_text = stringify_entity(row[field])
        if value_text:
            parts.append(f"{field}: {value_text}")
    return " | ".join(parts) if parts else "No entities extracted"

df["_extracted_summary"] = df.apply(
    lambda row: build_extraction_summary(row, EXTRACT_FIELDS),
    axis=1,
)

display(df[["review_text"] + EXTRACT_FIELDS + ["_extracted_summary"]].head(10))

In [ ]:
class ExtractEval(BaseModel):
    extraction_quality: Literal["keep", "refine"] = Field(
        description="Whether the extracted entities are ready to use as-is or should be refined."
    )
    refined_issue: str = Field(
        description="A concise issue phrase for downstream triage. Reuse or improve the extracted issue."
    )
    eval_reason: str = Field(description="Short explanation of the evaluation decision.")

EXTRACT_EVAL_PROMPT = dedent(
    """
    You are reviewing extracted entities from a customer review.

    <review>{review_text}</review>
    <extracted_entities>{_extracted_summary}</extracted_entities>

    Decide whether the extraction is ready for downstream triage.
    - Return extraction_quality='keep' when the extracted entities are specific and supported by the review.
    - Return extraction_quality='refine' when the issue is missing, vague, or can be improved.
    - Always return a concise issue phrase in refined_issue.
    - Keep eval_reason short and concrete.
    """
).strip()

df["extract_eval_json"] = df.ai.generate_response(
    prompt=EXTRACT_EVAL_PROMPT,
    is_prompt_template=True,
    response_format=ExtractEval,
)

parsed_extract_eval = df["extract_eval_json"].astype(str).apply(json.loads)
df["extract_eval"] = parsed_extract_eval.apply(lambda x: x.get("extraction_quality"))
df["refined_issue"] = parsed_extract_eval.apply(lambda x: x.get("refined_issue"))
df["extract_eval_reason"] = parsed_extract_eval.apply(lambda x: x.get("eval_reason"))

def choose_issue_for_triage(row):
    refined_value = str(row["refined_issue"]).strip()
    original_value = stringify_entity(row["issue"])
    if row["extract_eval"] == "refine" and refined_value:
        return refined_value
    return original_value if original_value else refined_value

df["issue_for_triage"] = df.apply(choose_issue_for_triage, axis=1)

display(
    df[
        [
            "review_text",
            "issue",
            "extract_eval",
            "refined_issue",
            "issue_for_triage",
            "extract_eval_reason",
        ]
    ].head(10)
)
display(df["extract_eval"].value_counts().rename_axis("extract_eval").reset_index(name="count"))

### 3.4 `ai.summarize`

Condense longer reviews into a format that is easier to scan during triage.

In [ ]:
df["summary"] = df["review_text"].ai.summarize()

display(df[["review_text", "summary"]].head(10))

In [ ]:
df["summary_char_count"] = df["summary"].astype(str).str.len()
valid = df[df["review_char_count"] > 0]

avg_original = valid["review_char_count"].mean() if len(valid) else 0
avg_summary = valid["summary_char_count"].mean() if len(valid) else 0
compression_pct = (1 - (avg_summary / avg_original)) * 100 if avg_original else 0

compression_table = pd.DataFrame(
    {
        "metric": ["avg_original_chars", "avg_summary_chars", "compression_percent"],
        "value": [round(avg_original, 1), round(avg_summary, 1), round(compression_pct, 1)],
    }
)
display(compression_table)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(
    ["original", "summary"],
    [avg_original, avg_summary],
    color=[FABRIC_BLUE, FABRIC_TEAL],
    edgecolor="white",
)
ax.set_title("Average review length vs summary length")
ax.set_ylabel("characters")
plt.tight_layout()
plt.show()

### 3.5 `ai.translate`

Translate summaries so the same workflow can support multilingual review processes.

In [ ]:
df["summary_spanish"] = df["summary"].ai.translate("spanish")

display(df[["summary", "summary_spanish"]].head(10))

### 3.6 `ai.fix_grammar`

Clean up raw review text when you need something easier to share with business users.

In [ ]:
df["review_corrected"] = df["review_text"].ai.fix_grammar()

display(df[["review_text", "review_corrected"]].head(10))

### 3.7 `ai.embed`

Generate embeddings for semantic search, clustering, and other vector-based workflows.

In [ ]:
df["embedding"] = df["review_text"].ai.embed()

embedding_overview = pd.DataFrame(
    {
        "metric": ["rows_with_embeddings", "embedding_dimension", "first_vector_preview"],
        "value": [
            int(df["embedding"].notna().sum()),
            len(df["embedding"].iloc[0]) if len(df) else 0,
            str(df["embedding"].iloc[0][:5]) if len(df) else "[]",
        ],
    }
)
display(embedding_overview)

### 3.8 `ai.similarity`

Rank reviews against a focused business question to surface the most relevant rows first.

In [ ]:
query = "durability and product quality issues"
df["similarity_score"] = df["review_text"].ai.similarity(query)

display(
    df[["review_text", "similarity_score"]]
    .sort_values("similarity_score", ascending=False)
    .head(10)
)

### 3.9 `ai.generate_response`

Turn the upstream signals into a structured support recommendation that is easy to operationalize.

In [ ]:
class ResponseSuggestion(BaseModel):
    reason: str = Field(description="Why this response strategy fits the review")
    response_type: Literal["thank_you", "apology", "follow_up", "no_response_needed"] = Field(
        description="Recommended response type"
    )
    priority: Literal["high", "medium", "low"] = Field(description="Support priority level")
    suggested_response: str = Field(description="Personalized 2-3 sentence response")

In [ ]:
RESPONSE_PROMPT = dedent(
    """
    You are a helpful customer support assistant.

    Review:
    <review>{review_text}</review>

    Detected sentiment: {sentiment}
    Detected topic: {topic}

    Return a warm and practical response to the customer.
    - Factor in the context of the review.
    - If sentiment is negative, apologize and include a concrete next step.
    - If sentiment is positive, thank the customer and reinforce value.
    - Keep the suggested response to 2-3 sentences.
    """
).strip()

df["response_json"] = df.ai.generate_response(
    prompt=RESPONSE_PROMPT,
    is_prompt_template=True,
    response_format=ResponseSuggestion,
)

display(df[["review_text", "response_json"]].head(10))

In [ ]:
parsed_response = df["response_json"].astype(str).apply(json.loads)
df["response_reason"] = parsed_response.apply(lambda x: x.get("reason"))
df["response_type"] = parsed_response.apply(lambda x: x.get("response_type"))
df["priority"] = parsed_response.apply(lambda x: x.get("priority"))
df["suggested_response"] = parsed_response.apply(lambda x: x.get("suggested_response"))

response_columns = ["response_reason", "response_type", "priority", "suggested_response"]
display(df[response_columns].head(10))

display(df["response_type"].value_counts().rename_axis("response_type").reset_index(name="count"))
display(df["priority"].value_counts().rename_axis("priority").reset_index(name="count"))

### 3.10 `ai.stats`

`ai.stats` is not an AI function itself. It is a helper that summarizes the AI Functions calls already run on the DataFrame, so you can review usage details such as request counts, success or failure totals, and token usage.

In [ ]:
display(df.ai.stats)

## 4. Build a triage dashboard

In [ ]:
pipeline_summary = pd.DataFrame(
    {
        "metric": [
            "rows_processed",
            "unique_sentiments",
            "unique_topics",
            "negative_reviews",
            "high_priority_items",
        ],
        "value": [
            len(df),
            df["sentiment"].nunique(),
            df["topic"].nunique(),
            int((df["sentiment"].astype(str) == "negative").sum()),
            int((df["priority"].astype(str) == "high").sum()),
        ],
    }
)

display(pipeline_summary)

display(
    df[df["priority"].astype(str) == "high"][
        ["review_text", "sentiment", "topic", "response_type", "suggested_response"]
    ].head(10)
)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

sentiment_dashboard = (
    df["sentiment"].astype(str).value_counts().reindex(["positive", "neutral", "mixed", "negative"])
    .fillna(0)
)
axes[0, 0].bar(
    sentiment_dashboard.index,
    sentiment_dashboard.values,
    color=[SENTIMENT_COLORS.get(value, FABRIC_BLUE) for value in sentiment_dashboard.index],
    edgecolor="white",
)
axes[0, 0].set_title("Sentiment")
axes[0, 0].set_ylabel("count")

topic_dashboard = df["topic"].astype(str).value_counts().head(8).sort_values()
axes[0, 1].barh(topic_dashboard.index, topic_dashboard.values, color=FABRIC_BLUE, edgecolor="white")
axes[0, 1].set_title("Top topics")
axes[0, 1].set_xlabel("count")

response_dashboard = df["response_type"].astype(str).value_counts()
axes[1, 0].bar(
    response_dashboard.index,
    response_dashboard.values,
    color=FABRIC_TEAL,
    edgecolor="white",
)
axes[1, 0].set_title("Response type")
axes[1, 0].set_ylabel("count")
axes[1, 0].tick_params(axis="x", rotation=20)

priority_dashboard = (
    df["priority"].astype(str).value_counts().reindex(["high", "medium", "low"]).fillna(0)
)
axes[1, 1].bar(
    priority_dashboard.index,
    priority_dashboard.values,
    color=[PRIORITY_COLORS.get(value, FABRIC_BLUE) for value in priority_dashboard.index],
    edgecolor="white",
)
axes[1, 1].set_title("Priority")
axes[1, 1].set_ylabel("count")

plt.tight_layout()
plt.show()

## 5. Interpreting results and next steps

### How to use this workflow
- Use **sentiment + topic** to identify where customer pain points are clustering.
- Use **extract** to pull substrings from text fields such as products, brands, and issues into downstream triage.
- Use **summary + translation + grammar fixes** to make reviews easier to read and share.
- Use **similarity + structured response suggestions** to prioritize repeated issues and accelerate support follow-up.

### Next steps
- Increase `SAMPLE_ROWS` once the notebook quality and cost look right.
- Replace the sample dataset with your own data.
- Move to the PySpark AI Functions when the dataset grows beyond what you want to keep in memory locally (for example, more than 1 GB).
- Use eval notebooks when you want systematic quality measurement instead of a starter walkthrough.

### Learn more
- Starter notebooks: [aka.ms/fabric-aifunctions-starter-notebooks](https://aka.ms/fabric-aifunctions-starter-notebooks)
- Eval notebooks: [aka.ms/fabric-aifunctions-eval-notebooks](https://aka.ms/fabric-aifunctions-eval-notebooks)
- AI Functions overview: [aka.ms/ai-functions](https://aka.ms/ai-functions)
- [Pandas AI Function Config Doc](https://learn.microsoft.com/en-us/fabric/data-science/ai-functions/pandas/configuration)
- Model catalog: [aka.ms/fabric-ai-models](https://aka.ms/fabric-ai-models)